In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import cv2
import random
from PIL import Image
from mtrain.utils import show, overlay_mask_on_img as OV, mkdir
from mtrain.disk import DiskBooleanMask as DBM, DiskImage as DI

In [ ]:
SAMPLE_DIR = Path("/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test")
dirs = sorted(SAMPLE_DIR.glob("*"))

In [ ]:
import argparse
from pathlib import Path

from mtrain.neg_mask.model.predict.full_image_8chan import (
    predict_and_return_probs,
)
from fastai.vision.all import (
    vision_learner,
    DataLoaders,
    resnet18,
    accuracy,
    F1Score,
    CrossEntropyLossFlat,
    ProgressCallback,
)
from tqdm import tqdm
import numpy as np
from mtrain.disk import DiskImage, DiskBooleanMask
from mtrain.neg_mask.model.learner import dummy_dls
from mtrain.neg_mask.model.crop_level_dataset import CropLevelDataset2Chan

NEG_MASK_MODEL_PATH = Path(
    "/Users/hariomnarang/Desktop/personal/roads/datasets/models/trash_classification/resnet18-size_130-chan_8-with_augs-iter_65-loss_focal"
)
MEDIUM_PAD = 130

def _load_neg_mask_model(model_path):
    LABELS = ["other", "trash"]
    DataLoaders.from_dsets(
        CropLevelDataset2Chan([], LABELS, True, medium_pad=MEDIUM_PAD),
        CropLevelDataset2Chan([], LABELS, False, medium_pad=MEDIUM_PAD),
    )

    learner = vision_learner(
        dummy_dls(LABELS),
        resnet18,
        n_in=8,
        metrics=[accuracy, F1Score(average="macro")],
        loss_func=CrossEntropyLossFlat(),
        n_out=len(LABELS),
        normalize=False,
    )
    learner = learner.remove_cb(ProgressCallback)

    learner = learner.load(NEG_MASK_MODEL_PATH)
    return learner

learner = _load_neg_mask_model(NEG_MASK_MODEL_PATH)

In [ ]:
from fastai.learner import Learner
from dataclasses import dataclass, field

@dataclass
class Arts:
    path: Path
    image: np.ndarray
    mask: np.ndarray
    m2: np.ndarray
    raw_probs: tuple[np.ndarray, np.ndarray] | None
    learner: Learner | None
    trash_probs: np.ndarray = field(init=False) 
    other_probs: np.ndarray = field(init=False) 

    def __post_init__(self):
        if self.raw_probs is not None:
            self.trash_probs, self.other_probs = self.raw_probs
        else:
            if self.learner is None:
                raise Exception("learner is none and raw probs is also none")
            else:
                self.trash_probs, self.other_probs = predict_and_return_probs(
                    self.image,
                    self.m2,
                    self.learner,
                    MEDIUM_PAD,
                )

    @property
    def trash_gt_other(self):
        return (self.trash_probs > self.other_probs)
    
    def trash_gt_thres(self, thres):
        return (self.trash_probs > thres)
    

def load_arts(p: Path) -> Arts:
    image = DI.load(p / "image.jpg")
    mask = DBM.load(p / "mask.png")
    m2 = DBM.load(p / "m2.png")
    return Arts(
        path=p, image=image, mask=mask, m2=m2, learner=learner, raw_probs=None
    )

In [ ]:
idx = 0

In [ ]:
bad_idxs = [
    5, 6, 7, 9, 10, 16, 20, 21, 31, 39, 41, 48, 49, 54
]

In [ ]:
i = 0

In [ ]:
idx = bad_idxs[i]
i += 1
print(idx)
# arts = load_arts(
#     Path(
#         "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/samples_mapillary/100/26210846388517270/"
#     )
# )
arts = load_arts(dirs[idx])
show([arts.image, OV(arts.image, arts.m2), OV(arts.image, arts.trash_gt_other)], (30,30), 3, axis="off")

In [ ]:
from mtrain.utils import draw_grid_cv2
show([draw_grid_cv2(arts.image[50:,50:], 100),], ncols=1 )

In [ ]:
from fastai.vision.all import load_learner, unet_learner
learner100 = load_learner(
    "/Users/hariomnarang/Desktop/gdrive-sync/garbage/experiments/enguled-bbox-levels-crops-v3/log/export_iter_14.pkl"
)

In [ ]:
from mtrain.smallnet.unet.predict.strided.single import strided_predict_unet_only_mask

In [ ]:
arts = load_arts(dirs[39])

In [ ]:
dirs[39]

In [ ]:
mask = strided_predict_unet_only_mask(arts.image, 100, learner100)

In [ ]:
plt.imshow(OV(arts.image, mask))

In [ ]:
learner100.model

In [ ]:
learner100.model.layers[0]._get_name()

In [ ]:
mylayer = None
for layer in learner100.model.layers:
    for layer in layer.children():
        if layer._get_name() == "Conv2d":
            mylayer = layer
            break
    if mylayer is not None:
        break

In [ ]:
learner.model[0][0]

In [ ]:
learner100.model[-1]

In [ ]:
import torch

# Grab the first layer weights: shape [64, 3, 7, 7]
# 1. Choose the layer (e.g., your first Conv2d)
first_conv_layer = learner100.model[0][0]

last_conv_layer = learner100.model[-2]
# last_conv_layer = list(learner100.model[4].children())[-2]

# 2. Create a container for the output
new_activations = {}
def get_activation(name):
    def hook(model, input, output):
        new_activations[name] = output.detach()
    return hook

_ = first_conv_layer.register_forward_hook(get_activation('first_layer'))
_ = last_conv_layer.register_forward_hook(get_activation('last_layer'))

# arts = load_arts(dirs[39])

# crop = torch.Tensor(arts.image[350:450, -135:-35]).permute([2,0,1]).unsqueeze(0)
# crop = torch.Tensor(arts.image[200:300, 400:500]).permute([2,0,1]).unsqueeze(0)
print(crop.shape)

with torch.no_grad():
    out = learner100.model(x[0])

In [ ]:
plt.imshow(learner100.predict(arts.image[350:450, -135:-35])[0])

In [ ]:
new_activations.keys()

In [ ]:
plt.imshow(arts.image[350:450, -135:-35])

In [ ]:
import cv2
# bright_img = cv2.convertScaleAbs(, alpha=1.1, beta=90)
# plt.imshow(bright_img)
hsv = cv2.cvtColor(arts.image[350:450, -135:-35], cv2.COLOR_BGR2HSV)
h, s, v = cv2.split(hsv)

# Add brightness to the 'V' channel
v = cv2.add(v, 50)

# Merge back and convert to BGR
final_hsv = cv2.merge((h, s, v))
bright_img = cv2.cvtColor(final_hsv, cv2.COLOR_HSV2BGR)

In [ ]:
plt.imshow(bright_img)

In [ ]:
plt.imshow(learner100.predict(bright_img)[0])

In [ ]:
learner100.model[-1]

In [ ]:
new_activations["first_layer"].shape, new_activations["last_layer"].shape

In [ ]:
out.shape

In [ ]:
dl = learner100.dls.test_dl([arts.image[350:450, -135:-35]])

In [ ]:
x = dl.one_batch()

In [ ]:
plt.imshow(learner100.loss_func.decodes(out)[0])

In [ ]:
plt.imshow(x[0][0].permute([1,2,0]))

In [ ]:
new_acts = [a.numpy() for a in new_activations["last_layer"][0]]
show(new_acts, ncols=2)

In [ ]:
new_acts = [a.numpy() for a in new_activations["first_layer"][0]]
show(new_acts, ncols=8)

In [ ]:
last_new_acts = [a.numpy() for a in new_activations["last_layer"][0]]
show(new_acts, ncols=8)

In [ ]:
learner.model

In [ ]:
# (64 filters, 3 RGB channels, 7x7 size)
kernels = learner.model[0][0].weight.detach().cpu()
print(kernels.shape)

# Plot the first 6 filters
fig, axes = plt.subplots(1, 6, figsize=(15, 5))
for i, ax in enumerate(axes):
    # Normalize the filter for display
    f = kernels[i][:3].permute(1, 2, 0) # Change to [7, 7, 3] for plotting
    print(f.shape)
    f = (f - f.min()) / (f.max() - f.min())
    ax.imshow(f)
    ax.axis('off')

In [ ]:
# Grab the first layer weights: shape [64, 3, 7, 7]
# (64 filters, 3 RGB channels, 7x7 size)
kernels = learner.model[0][0].weight.detach().cpu()
print(kernels.shape)

# Plot the first 6 filters
fig, axes = plt.subplots(1, 6, figsize=(15, 5))
for i, ax in enumerate(axes):
    # Normalize the filter for display
    f = kernels[i][3:6].permute(1, 2, 0) # Change to [7, 7, 3] for plotting
    print(f.shape)
    f = (f - f.min()) / (f.max() - f.min())
    ax.imshow(f)
    ax.axis('off')

In [ ]:
# Grab the first layer weights: shape [64, 3, 7, 7]
# (64 filters, 3 RGB channels, 7x7 size)
kernels = learner.model[0][0].weight.detach().cpu()
print(kernels.shape)

# Plot the first 6 filters
fig, axes = plt.subplots(1, 6, figsize=(15, 5))
for i, ax in enumerate(axes):
    # Normalize the filter for display
    f = kernels[i][6:7].permute(1, 2, 0) # Change to [7, 7, 3] for plotting
    print(f.shape)
    f = (f - f.min()) / (f.max() - f.min())
    ax.imshow(f)
    ax.axis('off')

In [ ]:
# Grab the first layer weights: shape [64, 3, 7, 7]
# (64 filters, 3 RGB channels, 7x7 size)
kernels = learner.model[0][0].weight.detach().cpu()
print(kernels.shape)

# Plot the first 6 filters
fig, axes = plt.subplots(1, 6, figsize=(15, 5))
for i, ax in enumerate(axes):
    # Normalize the filter for display
    f = kernels[i][7:8].permute(1, 2, 0) # Change to [7, 7, 3] for plotting
    print(f.shape)
    f = (f - f.min()) / (f.max() - f.min())
    ax.imshow(f)
    ax.axis('off')

In [ ]:
plt.imshow(layer.weight[12].permute([1,2,0]).detach().numpy())

In [ ]:
crop = arts.image[350:450, -135:-35]
print(crop.shape)
mask = learner100.predict(crop)[0].numpy()
plt.imshow(OV(crop, mask))

In [ ]:
arts.trash_probs.max()